# BS Blindspots - Exploratory Data Analysis

Investigating when and why Black-Scholes misprices options using real market data.

This notebook explores the feature matrices built by `src/features/build_features.py` for SPY and AAPL.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

FEATURES_DIR = Path('../data/features')

spy = pd.read_csv(FEATURES_DIR / 'SPY_features.csv')
aapl = pd.read_csv(FEATURES_DIR / 'AAPL_features.csv')

# Combine for joint analysis
df = pd.concat([spy, aapl], ignore_index=True)

print(f'SPY: {spy.shape}')
print(f'AAPL: {aapl.shape}')
print(f'Combined: {df.shape}')

## 2. Data Overview

In [ ]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print()
print('=== Dtypes ===')
print(df.dtypes)
print()
print('=== Null Counts ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'No nulls')
print()
print('=== Basic Statistics ===')
df.describe()

## 3. Target Distribution

How is BS mispricing distributed? Is it symmetric, or does BS tend to overprice/underprice?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Relative error distribution
axes[0].hist(df['rel_error'].clip(-2, 2), bins=80, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
axes[0].axvline(df['rel_error'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median={df["rel_error"].median():.4f}')
axes[0].set_xlabel('Relative Error (clipped to [-2, 2])')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of rel_error')
axes[0].legend()

# Signed error distribution
clip_val = df['signed_error'].quantile(0.99)
axes[1].hist(df['signed_error'].clip(-clip_val, clip_val), bins=80, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
axes[1].axvline(df['signed_error'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median={df["signed_error"].median():.4f}')
axes[1].set_xlabel('Signed Error (market - BS)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of signed_error')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'rel_error  - mean: {df["rel_error"].mean():.4f}, median: {df["rel_error"].median():.4f}, std: {df["rel_error"].std():.4f}')
print(f'signed_error - mean: {df["signed_error"].mean():.4f}, median: {df["signed_error"].median():.4f}')
print(f'Positive signed_error (BS underprices): {(df["signed_error"] > 0).mean():.1%}')
print(f'Negative signed_error (BS overprices):  {(df["signed_error"] < 0).mean():.1%}')

## 4. Error by Moneyness

Where is BS worst -- ITM, ATM, or OTM?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df.sample(min(5000, len(df)), random_state=42)

axes[0].scatter(sample['moneyness'], sample['rel_error'], alpha=0.15, s=5)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].axvline(1.0, color='gray', linestyle=':', linewidth=1, label='ATM (S/K=1)')
axes[0].set_xlabel('Moneyness (S/K)')
axes[0].set_ylabel('Relative Error')
axes[0].set_title('rel_error vs Moneyness')
axes[0].set_ylim(-2, 2)
axes[0].legend()

# Binned view
df['moneyness_bin'] = pd.cut(df['moneyness'], bins=[0.8, 0.9, 0.95, 0.98, 1.02, 1.05, 1.1, 1.2])
binned = df.groupby('moneyness_bin', observed=True)['rel_error'].agg(['mean', 'median', 'std', 'count'])
binned['mean'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Moneyness Bin')
axes[1].set_ylabel('Mean Relative Error')
axes[1].set_title('Mean rel_error by Moneyness Bin')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(binned.to_string())

## 5. Error by DTE

Do short-dated options have larger errors?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample['dte'], sample['rel_error'], alpha=0.15, s=5)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Days to Expiry')
axes[0].set_ylabel('Relative Error')
axes[0].set_title('rel_error vs DTE')
axes[0].set_ylim(-2, 2)

# Binned view
df['dte_bin'] = pd.cut(df['dte'], bins=[0, 7, 14, 30, 60, 90, 180])
dte_binned = df.groupby('dte_bin', observed=True)['rel_error'].agg(['mean', 'median', 'std', 'count'])
dte_binned['mean'].plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='black')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('DTE Bin')
axes[1].set_ylabel('Mean Relative Error')
axes[1].set_title('Mean rel_error by DTE Bin')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(dte_binned.to_string())
print(f'\nShort-dated (DTE <= 14) mean abs(rel_error): {df[df["dte"] <= 14]["rel_error"].abs().mean():.4f}')
print(f'Long-dated  (DTE > 14)  mean abs(rel_error): {df[df["dte"] > 14]["rel_error"].abs().mean():.4f}')

## 6. Error by Option Type

Are call vs put errors systematically different?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

calls = df[df['option_type'] == 'call']['rel_error']
puts = df[df['option_type'] == 'put']['rel_error']

bp = axes[0].boxplot(
    [calls.clip(-2, 2), puts.clip(-2, 2)],
    labels=['Call', 'Put'],
    patch_artist=True,
    showfliers=False
)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightsalmon')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_ylabel('Relative Error')
axes[0].set_title('rel_error by Option Type')

# Signed error comparison
bp2 = axes[1].boxplot(
    [df[df['option_type'] == 'call']['signed_error'].clip(-5, 5),
     df[df['option_type'] == 'put']['signed_error'].clip(-5, 5)],
    labels=['Call', 'Put'],
    patch_artist=True,
    showfliers=False
)
bp2['boxes'][0].set_facecolor('lightblue')
bp2['boxes'][1].set_facecolor('lightsalmon')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_ylabel('Signed Error ($)')
axes[1].set_title('signed_error by Option Type')

plt.tight_layout()
plt.show()

print(f'Calls - mean rel_error: {calls.mean():.4f}, median: {calls.median():.4f}, count: {len(calls)}')
print(f'Puts  - mean rel_error: {puts.mean():.4f}, median: {puts.median():.4f}, count: {len(puts)}')

## 7. Liquidity Analysis

Does liquidity (bid-ask spread) correlate with mispricing magnitude?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample['bid_ask_spread'], sample['abs_error'], alpha=0.15, s=5)
axes[0].set_xlabel('Bid-Ask Spread ($)')
axes[0].set_ylabel('Absolute Error ($)')
axes[0].set_title('abs_error vs Bid-Ask Spread')

# Relative spread vs relative error
axes[1].scatter(sample['bid_ask_rel'], sample['rel_error'].abs(), alpha=0.15, s=5, color='green')
axes[1].set_xlabel('Relative Bid-Ask Spread')
axes[1].set_ylabel('|Relative Error|')
axes[1].set_title('|rel_error| vs Relative Bid-Ask Spread')
axes[1].set_xlim(0, df['bid_ask_rel'].quantile(0.95))
axes[1].set_ylim(0, df['rel_error'].abs().quantile(0.95))

plt.tight_layout()
plt.show()

corr_abs = df['bid_ask_spread'].corr(df['abs_error'])
corr_rel = df['bid_ask_rel'].corr(df['rel_error'].abs())
print(f'Correlation: bid_ask_spread vs abs_error = {corr_abs:.4f}')
print(f'Correlation: bid_ask_rel vs |rel_error|  = {corr_rel:.4f}')

## 8. VIX Regime

Do high-vol environments have different error patterns?

Regime encoding: 0 = low (VIX < 15), 1 = medium (15-25), 2 = high (> 25)

In [ ]:
regime_labels = {0: 'Low (<15)', 1: 'Medium (15-25)', 2: 'High (>25)'}
df['vix_regime_label'] = df['vix_regime'].map(regime_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regime_groups = []
regime_names = []
for regime_val in sorted(df['vix_regime'].dropna().unique()):
    data = df[df['vix_regime'] == regime_val]['rel_error'].clip(-2, 2)
    regime_groups.append(data)
    regime_names.append(regime_labels.get(regime_val, str(regime_val)))

bp = axes[0].boxplot(regime_groups, labels=regime_names, patch_artist=True, showfliers=False)
colors = ['lightgreen', 'khaki', 'lightsalmon']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('VIX Regime')
axes[0].set_ylabel('Relative Error')
axes[0].set_title('rel_error by VIX Regime')

# Abs error by regime
abs_groups = []
for regime_val in sorted(df['vix_regime'].dropna().unique()):
    data = df[df['vix_regime'] == regime_val]['abs_error']
    abs_groups.append(data)

bp2 = axes[1].boxplot(abs_groups, labels=regime_names, patch_artist=True, showfliers=False)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
axes[1].set_xlabel('VIX Regime')
axes[1].set_ylabel('Absolute Error ($)')
axes[1].set_title('abs_error by VIX Regime')

plt.tight_layout()
plt.show()

print(df.groupby('vix_regime_label')['rel_error'].agg(['mean', 'median', 'std', 'count']).to_string())

## 9. Correlation Matrix

Heatmap of feature correlations to spot multicollinearity and interesting relationships.

In [ ]:
feature_cols = [
    'moneyness', 'log_moneyness', 'dte', 'time_to_maturity',
    'option_type_binary', 'hist_vol_20', 'hist_vol_60', 'vol_ratio',
    'vix', 'vol_of_vol', 'days_to_earnings', 'days_since_earnings',
    'in_earnings_window', 'bid_ask_spread', 'bid_ask_rel',
    'log_volume', 'log_open_interest', 'vix_regime',
    'rel_error', 'abs_error', 'signed_error'
]

available_cols = [c for c in feature_cols if c in df.columns]
corr = df[available_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(available_cols)))
ax.set_yticks(range(len(available_cols)))
ax.set_xticklabels(available_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(available_cols, fontsize=9)

for i in range(len(available_cols)):
    for j in range(len(available_cols)):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color=color)

plt.colorbar(im, ax=ax, shrink=0.8, label='Correlation')
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Top correlations with rel_error
if 'rel_error' in corr.columns:
    error_corr = corr['rel_error'].drop(['rel_error', 'abs_error', 'signed_error', 'sq_error'], errors='ignore')
    print('Top correlations with rel_error:')
    print(error_corr.abs().sort_values(ascending=False).head(10).to_string())

## 10. Key Takeaways

Run the notebook and fill in findings here based on the output above.

**Target Distribution:**
- Is mispricing symmetric or does BS have a directional bias (over/underprice)?
- What is the typical magnitude of rel_error?

**Moneyness:**
- Where is BS worst -- deep ITM, ATM, or OTM?
- Is the error pattern consistent with known BS limitations (vol smile)?

**DTE:**
- Do short-dated options (DTE < 14) show larger errors as hypothesized?
- Is there a DTE sweet spot where BS works best?

**Option Type:**
- Are puts systematically mispriced differently than calls?
- Does this align with the put skew / crash protection premium story?

**Liquidity:**
- How strong is the correlation between bid-ask spread and mispricing?
- Is mispricing partly a measurement artifact from illiquid options?

**VIX Regime:**
- Does high-vol regime produce qualitatively different errors?
- Is the BS underpricing of OTM puts in high-vol regimes visible?

**Feature Correlations:**
- Which features correlate most strongly with rel_error?
- Any multicollinearity concerns for the ML model?

---

*Findings to be filled in after running the notebook with real data.*